# Pairs Trading — Notebook 5: Live Signal Monitor (DRY-RUN)

**⚠️ این یه نوت‌بوک Dry-Run‌ـه — هیچ order واقعی فرستاده نمی‌شه.**

هدف: نشان دادن **سیگنال‌های فعلی** ۴ spread پرتفولیو (انتخاب‌شده در NB28) با دیتای زنده‌ی MT5:
- اتصال به MT5 (با graceful fallback به CSV اگه MT5 در دسترس نیست)
- fetch دیتای H4 اخیر برای ۸ سمبل
- refit β/α روی آخرین ۱۲ ماه (مطابق walk-forward NB27)
- محاسبه‌ی z-score فعلی برای هر spread
- تشخیص action (open_long / open_short / exit / hold)
- محاسبه‌ی lot size با β-scaling و risk budget
- چاپ report: "اگه live بود، این ordersها رو می‌فرستادم"

## برای production-grade (نه این نوت‌بوک)
این نوت‌بوک فقط signal monitor‌ـه. برای trading واقعی باید روی `mt5/run_multi_scalper.py` pattern یک `mt5/run_pairs_trading.py` ساخته بشه که از infrastructure موجود استفاده کنه:
- `execution/ExecutionEngine` — اوردر sender + retry + cooldown
- `execution/mt5_watchdog` — reconnect خودکار
- `execution/structured_logger` — لاگ JSON
- `telegram_bot/Mt5Notifier` — alerts

**نکته‌ی مهم درباره‌ی pairs trading vs single-symbol scalper موجود:**
ExecutionEngine برای single-symbol طراحی شده. برای pairs باید یه wrapper بنویسیم که:
1. دو leg همزمان open کنه (long y، short x با volume_y × β / quote_ratio)
2. partial fill protection: اگر یکی open شد و دیگری rejected، اولی فوراً close بشه
3. exit همزمان هر دو leg در سیگنال exit

In [ ]:
from __future__ import annotations
import os
import sys
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Try to import MetaTrader5 — graceful fallback if not available (Linux dev box, no MT5 terminal)
try:
    import MetaTrader5 as mt5
    HAVE_MT5 = True
except ImportError:
    HAVE_MT5 = False
    print("⚠️  MetaTrader5 module not importable — will fall back to CSV cache.")

print(f"HAVE_MT5: {HAVE_MT5}")
print("ready")

## Configuration

In [ ]:
TF                   = "H4"           # portfolio TF chosen in NB28
TRAIN_MONTHS         = 12             # β refit window (mirror NB27)
Z_WINDOW_BARS        = 200            # rolling z-window (same as default in NB26/27)
ENTRY_Z              = 2.0
EXIT_Z               = 0.5
STOP_Z               = 4.0
RISK_PER_LEG_PCT     = 0.5            # % of account equity at risk per leg
ACCOUNT_EQUITY_USD   = 10_000.0       # fallback if MT5 not connected

STAT_DIR = PROJECT_ROOT / "notebooks" / "data" / "stat_arb"
DATA_DIR = PROJECT_ROOT / "notebooks" / "data"
REAL_TZ  = "Europe/Nicosia"

# MT5 connection — credentials from env vars (never hardcode)
MT5_LOGIN     = os.environ.get("MT5_LOGIN")
MT5_PASSWORD  = os.environ.get("MT5_PASSWORD")
MT5_SERVER    = os.environ.get("MT5_SERVER")
MT5_PATH      = os.environ.get("MT5_TERMINAL_PATH")

print(f"TF={TF}  z_window={Z_WINDOW_BARS}  entry=±{ENTRY_Z}σ  exit=±{EXIT_Z}σ  stop=±{STOP_Z}σ")
print(f"risk per leg: {RISK_PER_LEG_PCT}%  fallback equity: ${ACCOUNT_EQUITY_USD:,.0f}")

## ۱) Load portfolio (4 spreads from NB28)

In [ ]:
portfolio = pd.read_csv(STAT_DIR / f"portfolio_selected_{TF}.csv")
print(f"portfolio ({TF}, {len(portfolio)} spreads):")
print(portfolio.to_string(index=False))

symbols_needed = sorted(set(portfolio["y"]).union(portfolio["x"]))
print(f"\nsymbols needed: {symbols_needed}")

## ۲) MT5 connection + data fetcher (with CSV fallback)

In [ ]:
MT5_CONNECTED = False

def connect_mt5() -> bool:
    """Try mt5.initialize() with optional credentials. Returns success bool."""
    global MT5_CONNECTED
    if not HAVE_MT5:
        return False
    kwargs = {}
    if MT5_PATH:     kwargs["path"]     = MT5_PATH
    if MT5_LOGIN:    kwargs["login"]    = int(MT5_LOGIN)
    if MT5_PASSWORD: kwargs["password"] = MT5_PASSWORD
    if MT5_SERVER:   kwargs["server"]   = MT5_SERVER
    try:
        ok = mt5.initialize(**kwargs)
    except Exception as e:
        print(f"  mt5.initialize raised: {e}")
        return False
    if not ok:
        print(f"  mt5.initialize failed: {mt5.last_error()}")
        return False
    MT5_CONNECTED = True
    info = mt5.account_info()
    if info:
        print(f"  connected: login={info.login}  server={info.server}  equity={info.equity:.2f} {info.currency}")
    return True

connect_mt5()
print(f"MT5_CONNECTED: {MT5_CONNECTED}")

In [ ]:
TF_MT5 = {"H1": getattr(mt5, "TIMEFRAME_H1", 16385) if HAVE_MT5 else None,
          "H4": getattr(mt5, "TIMEFRAME_H4", 16388) if HAVE_MT5 else None}

def fetch_h4_close_live(symbol: str, n_bars: int) -> pd.Series:
    """Fetch recent H4 close prices via MT5. Raises on failure."""
    rates = mt5.copy_rates_from_pos(symbol, TF_MT5[TF], 0, n_bars)
    if rates is None or len(rates) == 0:
        raise RuntimeError(f"copy_rates_from_pos returned None for {symbol}: {mt5.last_error()}")
    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True).dt.tz_convert(REAL_TZ)
    return pd.Series(df["close"].values, index=df["time"], name=symbol)

def fetch_h4_close_csv(symbol: str, n_bars: int) -> pd.Series:
    """Fallback: read H1 CSV cache, resample to TF (MT5-compatible labeling)."""
    path = DATA_DIR / symbol / "H1" / "ohlcv.csv"
    df = pd.read_csv(path, parse_dates=["time"])
    naive = df["time"].dt.tz_localize(None)
    ts = naive.dt.tz_localize(REAL_TZ, ambiguous="NaT", nonexistent="NaT")
    s = pd.Series(df["close"].values, index=ts, name=symbol).dropna().sort_index()
    if TF != "H1":
        rule = {"H4": "4h", "D1": "1D"}[TF]
        # MT5 default: label='left', closed='left' (bar label = open time)
        s = s.resample(rule, label="left", closed="left").last().dropna()
    return s.iloc[-n_bars:]

bars_per_year = {"H1": 24*252, "H4": 6*252}[TF]
N_BARS = TRAIN_MONTHS * bars_per_year // 12 + Z_WINDOW_BARS + 50
print(f"fetching {N_BARS} bars per symbol ({TF})")

# Strategy: try MT5 for ALL symbols first; if ANY fail, fall back to CSV for ALL
# (mixing sources gives misaligned timestamps → empty inner-join).
used_source = "none"
prices: dict[str, pd.Series] = {}
if MT5_CONNECTED:
    live_ok = True
    for sym in symbols_needed:
        try:
            prices[sym] = fetch_h4_close_live(sym, N_BARS)
        except Exception as e:
            print(f"  [{sym}] live fetch failed ({e})")
            live_ok = False
            break
    if live_ok:
        used_source = "MT5"
    else:
        print("  → at least one symbol failed live fetch; using CSV for all (consistent timestamps)")
        prices = {}
if not prices:
    prices = {sym: fetch_h4_close_csv(sym, N_BARS) for sym in symbols_needed}
    used_source = "CSV"

df_prices = pd.concat(prices, axis=1, sort=True).dropna()
print(f"\nsource: {used_source}")
print(f"aligned prices: {df_prices.shape}")
print(f"range: {df_prices.index.min()} → {df_prices.index.max()}")
df_prices.tail(3)

## ۳) Compute current signal per spread

In [ ]:
@dataclass
class SpreadSignal:
    y: str; x: str
    alpha: float; beta: float
    spread_now: float
    z_now: float
    action: str          # 'open_long', 'open_short', 'exit', 'hold'
    reason: str          # explanation


def signal_for_spread(y_sym: str, x_sym: str, prices_df: pd.DataFrame) -> SpreadSignal:
    py, px = prices_df[y_sym], prices_df[x_sym]
    # Use the last TRAIN_MONTHS*bars_per_year bars as training for β/α (mirror NB27)
    train_bars = TRAIN_MONTHS * bars_per_year // 12
    train_y = np.log(py.iloc[-train_bars - Z_WINDOW_BARS:-Z_WINDOW_BARS].values)
    train_x = np.log(px.iloc[-train_bars - Z_WINDOW_BARS:-Z_WINDOW_BARS].values)
    X = np.column_stack([np.ones_like(train_x), train_x])
    coef, *_ = np.linalg.lstsq(X, train_y, rcond=None)
    alpha, beta = float(coef[0]), float(coef[1])

    log_y = np.log(py)
    log_x = np.log(px)
    spread = log_y - beta * log_x - alpha
    mu = spread.rolling(Z_WINDOW_BARS).mean()
    sd = spread.rolling(Z_WINDOW_BARS).std()
    z = (spread - mu) / sd
    z_now = float(z.iloc[-1])
    spread_now = float(spread.iloc[-1])

    if z_now > ENTRY_Z:
        action, reason = "open_short", f"z={z_now:+.2f} > +{ENTRY_Z} → short spread (short {y_sym}, long {x_sym})"
    elif z_now < -ENTRY_Z:
        action, reason = "open_long",  f"z={z_now:+.2f} < -{ENTRY_Z} → long spread (long {y_sym}, short {x_sym})"
    elif abs(z_now) <= EXIT_Z:
        action, reason = "exit_or_hold", f"z={z_now:+.2f} inside ±{EXIT_Z} → close any open position (else hold)"
    elif abs(z_now) >= STOP_Z:
        action, reason = "stop", f"z={z_now:+.2f} beyond ±{STOP_Z} → stop-out"
    else:
        action, reason = "hold", f"z={z_now:+.2f} between ±{EXIT_Z} and ±{ENTRY_Z} → hold"
    return SpreadSignal(y=y_sym, x=x_sym, alpha=alpha, beta=beta,
                        spread_now=spread_now, z_now=z_now,
                        action=action, reason=reason)

signals = [signal_for_spread(r["y"], r["x"], df_prices) for _, r in portfolio.iterrows()]
sig_df = pd.DataFrame([s.__dict__ for s in signals])
print("=== current signals ===")
print(sig_df[["y", "x", "beta", "z_now", "action", "reason"]].to_string(index=False))

## ۴) Position sizing (β-scaled, risk-per-leg budget)

**فرمول ساده:**
- size_y (lots) از `RISK_PER_LEG_PCT × equity / (estimated_stop_pip × pip_value)` (تخمین stop = 1% adverse move)
- size_x = size_y × β × (quote_ratio) — برای neutralize کردن exposure ارز مشترک

این یه **تخمین حداقلی**ست. نسخه‌ی production نیاز به symbol_info دقیق هر symbol از MT5 (contract_size, tick_value, tick_size) داره — که از طریق `mt5.symbol_info(s)` می‌گیریم.

In [ ]:
def get_equity_usd() -> float:
    if MT5_CONNECTED:
        info = mt5.account_info()
        if info:
            return float(info.equity)
    return ACCOUNT_EQUITY_USD

def get_contract_size(sym: str) -> float:
    """Standard FX contract = 100,000 of base currency. Override via MT5 if connected."""
    if MT5_CONNECTED:
        si = mt5.symbol_info(sym)
        if si:
            return float(si.trade_contract_size)
    return 100_000.0

def get_volume_bounds(sym: str) -> tuple[float, float, float]:
    """Return (min, step, max) lot size from MT5 symbol_info, with sane fallback."""
    if MT5_CONNECTED:
        si = mt5.symbol_info(sym)
        if si:
            return float(si.volume_min), float(si.volume_step), float(si.volume_max)
    return 0.01, 0.01, 100.0

def round_to_step(x: float, step: float) -> float:
    return round(round(x / step) * step, 8)

def estimate_lot_sizes(sig: SpreadSignal, equity_usd: float,
                       assumed_stop_pct: float = 0.01) -> tuple[float, float, str]:
    """Return (lots_y, lots_x, note). Sizes respect broker volume_min/step.

    Approximate sizing: an `assumed_stop_pct` adverse move on the y-leg
    should cost `risk_usd`. x-leg sized for β × y-notional.
    """
    risk_usd = equity_usd * (RISK_PER_LEG_PCT / 100.0)
    px_y = float(df_prices[sig.y].iloc[-1])
    px_x = float(df_prices[sig.x].iloc[-1])
    cs_y = get_contract_size(sig.y)
    cs_x = get_contract_size(sig.x)
    vmin_y, vstep_y, vmax_y = get_volume_bounds(sig.y)
    vmin_x, vstep_x, vmax_x = get_volume_bounds(sig.x)

    notional_y_per_lot = cs_y * px_y
    raw_lots_y = risk_usd / (notional_y_per_lot * assumed_stop_pct)

    note = ""
    if raw_lots_y < vmin_y:
        note = f"raw lots_y={raw_lots_y:.4f} below broker min {vmin_y} → clamped (UNDER-RISK)"
        lots_y = vmin_y
    else:
        lots_y = round_to_step(raw_lots_y, vstep_y)

    notional_y = lots_y * notional_y_per_lot
    target_notional_x = abs(sig.beta) * notional_y
    raw_lots_x = target_notional_x / (cs_x * px_x)
    if raw_lots_x < vmin_x:
        lots_x = vmin_x
        if not note:
            note = f"raw lots_x={raw_lots_x:.4f} below broker min {vmin_x} → clamped"
    else:
        lots_x = round_to_step(raw_lots_x, vstep_x)

    return lots_y, lots_x, note

equity = get_equity_usd()
print(f"equity used for sizing: ${equity:,.2f}  (risk per leg: ${equity*RISK_PER_LEG_PCT/100:.2f})")
sig_df["lots_y"] = 0.0
sig_df["lots_x"] = 0.0
sig_df["size_note"] = ""
for s in signals:
    lots_y, lots_x, note = estimate_lot_sizes(s, equity)
    mask = (sig_df["y"] == s.y) & (sig_df["x"] == s.x)
    sig_df.loc[mask, "lots_y"] = lots_y
    sig_df.loc[mask, "lots_x"] = lots_x
    sig_df.loc[mask, "size_note"] = note
print("\n=== sized signals ===")
print(sig_df[["y", "x", "beta", "z_now", "action", "lots_y", "lots_x", "size_note"]].to_string(index=False))

## ۵) Dry-run order report

نمایش دقیق orders که اگه live بود فرستاده می‌شد. **هیچ تماس واقعی به broker انجام نمی‌شه.**

In [ ]:
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"=== DRY-RUN REPORT @ {now} ===\n")

actionable = sig_df[sig_df["action"].isin(["open_long", "open_short", "stop", "exit_or_hold"])]
if actionable.empty:
    print("no actionable signals; all spreads in hold range.")
else:
    for _, r in actionable.iterrows():
        print(f"  spread {r['y']} ~ {r['x']}  (β={r['beta']:+.3f})")
        print(f"    z_now = {r['z_now']:+.3f}")
        print(f"    action = {r['action']}")
        if r["action"] == "open_long":
            print(f"    → would send: BUY  {r['y']:>7s} {r['lots_y']:>5.2f} lots")
            print(f"    → would send: SELL {r['x']:>7s} {r['lots_x']:>5.2f} lots")
        elif r["action"] == "open_short":
            print(f"    → would send: SELL {r['y']:>7s} {r['lots_y']:>5.2f} lots")
            print(f"    → would send: BUY  {r['x']:>7s} {r['lots_x']:>5.2f} lots")
        elif r["action"] in ("exit_or_hold", "stop"):
            print(f"    → would CLOSE any existing position on this pair")
        print()

hold_count = (sig_df["action"] == "hold").sum()
print(f"({hold_count} spreads in hold; total {len(sig_df)})")

## ۶) Save snapshot + cleanup

In [ ]:
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
snap_dir = STAT_DIR / "live_signals"
snap_dir.mkdir(exist_ok=True, parents=True)
snap_path = snap_dir / f"signals_{TF}_{ts}.csv"
sig_df.to_csv(snap_path, index=False)
print(f"saved snapshot → {snap_path}")

if MT5_CONNECTED:
    mt5.shutdown()
    print("mt5.shutdown() called.")

print("\nProduction-ready next steps:")
print("  1. Build mt5/run_pairs_trading.py mirroring mt5/run_multi_scalper.py:")
print("     - load portfolio_selected_H4.csv")
print("     - loop on H4 close (sleep until next 4h boundary)")
print("     - re-fit β/α each cycle (cheap; ~50ms per spread)")
print("     - for each spread with actionable signal, fire BOTH legs via ExecutionEngine")
print("  2. Wrap two-leg order in a PairsExecutionAdapter:")
print("     - submit y-leg via ExecutionEngine")
print("     - if y filled, submit x-leg (else abort)")
print("     - if x submission fails, immediately close y")
print("     - persist {pair → leg_tickets} state for exit matching")
print("  3. Run with --dry-run for ≥2 weeks against a demo account")
print("     before any live capital.")